# 1- Neural Network Design in Pytorch

In [13]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision.datasets as datasets
import torchvision.transforms as transforms

from torch.nn.functional import conv2d, max_pool2d, cross_entropy

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.4.0


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [15]:
plt.rc("figure", dpi=100)

batch_size = 100
n_epochs = 100

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5,), std=(0.5,))
])

In [16]:
train_dataset = datasets.MNIST(
    "./data",
    download=True,
    train=True,
    transform=transform,
)

test_dataset = datasets.MNIST(
    "./data",
    download=True,
    train=False,
    transform=transform,
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Training samples: 60000
Test samples: 10000


In [17]:
def init_weights(shape):
    std = np.sqrt(2. / shape[0])
    w = torch.randn(size=shape, device=device) * std
    w.requires_grad = True
    return w

def rectify(x):
    return torch.max(torch.zeros_like(x), x)

In [18]:
class RMSprop(optim.Optimizer):
    def __init__(self, params, lr=1e-3, alpha=0.5, eps=1e-8):
        defaults = dict(lr=lr, alpha=alpha, eps=eps)
        super(RMSprop, self).__init__(params, defaults)

    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                grad = p.grad.data
                state = self.state[p]

                if len(state) == 0:
                    state['square_avg'] = torch.zeros_like(p.data)

                square_avg = state['square_avg']
                alpha = group['alpha']

                square_avg.mul_(alpha).addcmul_(grad, grad, value=1 - alpha)
                avg = square_avg.sqrt().add_(group['eps'])

                p.data.addcdiv_(grad, avg, value=-group['lr'])

In [19]:
def model(x, w_h, w_h2, w_o):
    h = rectify(x @ w_h)
    h2 = rectify(h @ w_h2)
    pre_softmax = h2 @ w_o
    return pre_softmax

In [20]:
w_h = init_weights((784, 625))
w_h2 = init_weights((625, 625))
w_o = init_weights((625, 10))

optimizer = RMSprop(params=[w_h, w_h2, w_o])

print("Weights initialized for Task 1 (Plain ReLU)")

Weights initialized for Task 1 (Plain ReLU)


In [21]:
n_epochs = 100


In [22]:
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")

2.4.0
True
NVIDIA GeForce RTX 4070 Laptop GPU


In [ ]:
train_loss_plain = []
test_loss_plain = []
eval_epochs_plain = []

def evaluate_plain_loss(dataloader):
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device).reshape(x.shape[0], 784)
            y = y.to(device)
            logits = model(x, w_h, w_h2, w_o)
            loss = cross_entropy(logits, y, reduction="sum")
            total_loss += float(loss)
            total_samples += y.shape[0]
    return total_loss / total_samples

print("Training Task 1 (Plain ReLU)...")
for epoch in range(n_epochs + 1):
    for idx, batch in enumerate(train_dataloader):
        x, y = batch
        x = x.to(device).reshape(x.shape[0], 784)
        y = y.to(device)
        
        logits = model(x, w_h, w_h2, w_o)
        optimizer.zero_grad()
        loss = cross_entropy(logits, y, reduction="mean")
        loss.backward()
        optimizer.step()
    
    if epoch % 10 == 0:
        train_loss_plain.append(evaluate_plain_loss(train_dataloader))
        test_loss_plain.append(evaluate_plain_loss(test_dataloader))
        eval_epochs_plain.append(epoch)
        print(f"Epoch {epoch:3d}: Train Loss = {train_loss_plain[-1]:.2e}, Test Loss = {test_loss_plain[-1]:.2e}")

plt.figure(figsize=(8, 5))
plt.plot(eval_epochs_plain, train_loss_plain, 'o-', label="Train (Plain)", markersize=4)
plt.plot(eval_epochs_plain, test_loss_plain, 's-', label="Test (Plain)", markersize=4)
plt.title("Task 1: Plain ReLU - Train and Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

Training Task 1 (Plain ReLU)...
Epoch   0: Train Loss = 2.01e-01, Test Loss = 1.96e-01


#### from the test_loss and train_lost figure, it is obvious that the model is overfitting. The train loss is decreasing, but the test loss is increasing after a certain point. This indicates that the model is learning the training data too well and is not generalizing to unseen data.

> we intend to reduce this overfitting and solve the issue by using dropout technique

# 2- Dropout

In [ ]:
def dropout(X, p_drop=0.5):
    if p_drop == 0:
        return X
    p = min(max(p_drop, 0.0), 1.0)
    mask = (torch.rand_like(X) > p).float()
    output = X * mask / (1.0 - p)
    return output

In [ ]:
def evaluate_loss_dropout(dataloader):
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device).reshape(x.shape[0], 784)
            y = y.to(device)
            logits = dropout_model(x, w_h, w_h2, w_o,
                                   p_drop_input=0.0, p_drop_hidden=0.0)
            loss = cross_entropy(logits, y, reduction="sum")
            total_loss += float(loss)
            total_samples += y.shape[0]
    return total_loss / total_samples

for epoch in range(n_epochs + 1):
    for idx, batch in enumerate(train_dataloader):
        x, y = batch
        x = x.to(device).reshape(x.shape[0], 784)
        y = y.to(device)
        
        logits = dropout_model(x, w_h, w_h2, w_o,
                               p_drop_input=0.2, p_drop_hidden=0.5)
        optimizer.zero_grad()
        loss = cross_entropy(logits, y, reduction="mean")
        loss.backward()
        optimizer.step()
    
    if epoch % 10 == 0:
        train_loss_dropout.append(evaluate_loss_dropout(train_dataloader))
        test_loss_dropout.append(evaluate_loss_dropout(test_dataloader))
        eval_epochs_dropout.append(epoch)
        print(f"Epoch {epoch:3d}: Train Loss = {train_loss_dropout[-1]:.2e}, Test Loss = {test_loss_dropout[-1]:.2e}")

plt.plot(eval_epochs_dropout, train_loss_dropout, label="Train (Dropout)")
plt.plot(eval_epochs_dropout, test_loss_dropout, label="Test (Dropout)")
plt.title("Dropout Model – Train and Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

As it is visible I wrote dropout function and used it in my training loop but used it on x (just first input) so it will effect first layer and the rest would cause again overfitting. 

> I will solve this in the leading cell by converting model function that was written to dropout

In [ ]:
def dropout_model(X, w_h, w_h2, w_o, p_drop_input, p_drop_hidden):
    # input dropout
    if p_drop_input > 0:
        mask = (torch.rand_like(X) > p_drop_input).float()
        X = X * mask / (1.0 - p_drop_input)
    
    # first hidden layer
    h = rectify(X @ w_h)
    if p_drop_hidden > 0:
        mask = (torch.rand_like(h) > p_drop_hidden).float()
        h = h * mask / (1.0 - p_drop_hidden)
    
    # second hidden layer
    h2 = rectify(h @ w_h2)
    if p_drop_hidden > 0:
        mask = (torch.rand_like(h2) > p_drop_hidden).float()
        h2 = h2 * mask / (1.0 - p_drop_hidden)
    
    pre_softmax = h2 @ w_o
    return pre_softmax

In [ ]:
# Fresh weights for dropout experiment
w_h = init_weights((784, 625))
w_h2 = init_weights((625, 625))
w_o = init_weights((625, 10))
optimizer = RMSprop(params=[w_h, w_h2, w_o])

# Containers for dropout results
train_loss_dropout = []
test_loss_dropout = []
eval_epochs_dropout = []

print("Weights reinitialized for dropout training.")

In [ ]:
def evaluate_dropout_loss(dataloader):
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device).reshape(x.shape[0], 784)
            y = y.to(device)
            logits = dropout_model(x, w_h, w_h2, w_o,
                                   p_drop_input=0.0, p_drop_hidden=0.0)
            loss = cross_entropy(logits, y, reduction="sum")
            total_loss += float(loss)
            total_samples += y.shape[0]
    return total_loss / total_samples

print("Training Task 2 (Dropout)...")
for epoch in range(n_epochs + 1):
    for idx, batch in enumerate(train_dataloader):
        x, y = batch
        x = x.to(device).reshape(x.shape[0], 784)
        y = y.to(device)
        logits = dropout_model(x, w_h, w_h2, w_o,
                               p_drop_input=0.2, p_drop_hidden=0.5)
        optimizer.zero_grad()
        loss = cross_entropy(logits, y, reduction="mean")
        loss.backward()
        optimizer.step()
    
    if epoch % 10 == 0:
        train_loss_dropout.append(evaluate_dropout_loss(train_dataloader))
        test_loss_dropout.append(evaluate_dropout_loss(test_dataloader))
        eval_epochs_dropout.append(epoch)
        print(f"Epoch {epoch:3d}: Train Loss = {train_loss_dropout[-1]:.2e}, Test Loss = {test_loss_dropout[-1]:.2e}")

plt.figure(figsize=(8, 5))
plt.plot(eval_epochs_dropout, train_loss_dropout, 'o-', label="Train (Dropout)", markersize=4)
plt.plot(eval_epochs_dropout, test_loss_dropout, 's-', label="Test (Dropout)", markersize=4)
plt.title("Task 2: Dropout Model - Train and Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

### why we use dropout:

fully connected layer -> large number of neurons -> co-adaption 


co-adaption -> when multiple neurongs in a layer extract the same or very similar features


When the connection weights for 2 different neurons are nearly identical
	
#### problems:
Wastage of machine's resources when computing the same output.

if more neurons are extracting the same features, it adds more significance to those features for our model. 
		
		
This will lead to overfitting ->> if the duplicate extracted features are specific to only the training set.
		
	
#### mitigation: 
we minimize co-adaption by using dropout
	
	
#### functionality:
we randomly shutdown some fractions of a layer's neurons at each training step 


-> zero out the neuron value.
		
#### dropout rate:
The fraction of neurons to be zeroed out (rd)


the remaining neurons have their values multiplied by 1/1-rd


then the overall sums of neurons values will be the same
	
	
#### benefits:
By using dropout, in every iteration, we work on a smaller neural network 


-> approaches regularization
		


Dropout helps in shrinking the squared norm of the weights


-> leads to reduction of overfitting

**RELU**

In [ ]:
def prelu(x, a):
    """Parametric ReLU: x if x > 0, else a * x"""
    return torch.where(x > 0, x, a * x)

def prelu_model(x, w_h, w_h2, w_o, a1, a2):
    h = prelu(x @ w_h, a1)
    h2 = prelu(h @ w_h2, a2)
    pre_softmax = h2 @ w_o
    return pre_softmax

In [ ]:
# Reset weights for fair comparison
w_h = init_weights((784, 625))
w_h2 = init_weights((625, 625))
w_o = init_weights((625, 10))

# Learnable parameters for PReLU (initialized to 0.25)
a1 = torch.tensor(0.25, device=device, requires_grad=True)
a2 = torch.tensor(0.25, device=device, requires_grad=True)

optimizer = RMSprop(params=[w_h, w_h2, w_o, a1, a2])

print("Weights and PReLU parameters initialized.")
print(f"a1 initial: {a1.item():.3f}, a2 initial: {a2.item():.3f}")

In [ ]:
def evaluate_prelu_loss(dataloader):
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device).reshape(x.shape[0], 784)
            y = y.to(device)
            logits = prelu_model(x, w_h, w_h2, w_o, a1, a2)
            loss = cross_entropy(logits, y, reduction="sum")
            total_loss += float(loss)
            total_samples += y.shape[0]
    return total_loss / total_samples

***Training with RELU***

In [ ]:
train_loss_prelu = []
test_loss_prelu = []
eval_epochs_prelu = []

def evaluate_prelu_loss(dataloader):
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device).reshape(x.shape[0], 784)
            y = y.to(device)
            logits = prelu_model(x, w_h, w_h2, w_o, a1, a2)
            loss = cross_entropy(logits, y, reduction="sum")
            total_loss += float(loss)
            total_samples += y.shape[0]
    return total_loss / total_samples

print("Training Task 3 (Parametric ReLU)...")
for epoch in range(n_epochs + 1):
    for idx, batch in enumerate(train_dataloader):
        x, y = batch
        x = x.to(device).reshape(x.shape[0], 784)
        y = y.to(device)
        logits = prelu_model(x, w_h, w_h2, w_o, a1, a2)
        optimizer.zero_grad()
        loss = cross_entropy(logits, y, reduction="mean")
        loss.backward()
        optimizer.step()
    
    if epoch % 10 == 0:
        train_loss_prelu.append(evaluate_prelu_loss(train_dataloader))
        test_loss_prelu.append(evaluate_prelu_loss(test_dataloader))
        eval_epochs_prelu.append(epoch)
        print(f"Epoch {epoch:3d}: Train Loss = {train_loss_prelu[-1]:.2e}, Test Loss = {test_loss_prelu[-1]:.2e}, a1 = {a1.item():.3f}, a2 = {a2.item():.3f}")

plt.figure(figsize=(8, 5))
plt.plot(eval_epochs_prelu, train_loss_prelu, 'o-', label="Train (PReLU)", markersize=4)
plt.plot(eval_epochs_prelu, test_loss_prelu, 's-', label="Test (PReLU)", markersize=4)
plt.title("Task 3: Parametric ReLU - Train and Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

***Lets compare***

In [ ]:
# Note: You need to have saved the final losses from Task 1 and Task 2.
# If you didn't, you can re-evaluate or use the last values from your previous runs.

print("\n========== FINAL COMPARISON ==========")
print(f"Task 1 (Plain ReLU)         - Test Loss: {test_loss[-1]:.2e}")
print(f"Task 2 (Dropout)            - Test Loss: {test_loss_dropout_correct[-1]:.2e}")
print(f"Task 3 (Parametric ReLU)    - Test Loss: {test_loss_prelu[-1]:.2e}")
print("=====================================")

# Optional: Plot all together
plt.figure(figsize=(10, 6))
plt.plot(eval_epochs, test_loss, 'o-', label="Plain ReLU", markersize=4)
plt.plot(eval_epochs_dropout, test_loss_dropout_correct, 's-', label="Dropout", markersize=4)
plt.plot(eval_epochs_prelu, test_loss_prelu, '^-', label="Parametric ReLU", markersize=4)
plt.title("Test Loss Comparison Across Models")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

**Task 4**

In [ ]:
def conv_output_size(input_size, kernel, padding=0, stride=1):
    return (input_size - kernel + 2*padding) // stride + 1

after_conv1 = conv_output_size(28, 5)
after_pool1 = after_conv1 // 2
after_conv2 = conv_output_size(after_pool1, 5)
after_pool2 = after_conv2 // 2
after_conv3 = conv_output_size(after_pool2, 3)
after_pool3 = after_conv3 // 2

flat_features = 128 * after_pool3 * after_pool3
print(f"Flattened feature size: {flat_features}")

In [ ]:
def conv_model(x, conv_weights, fc_weights, dropout_probs):
    p_conv1, p_conv2, p_conv3, p_fc = dropout_probs
    
    x = rectify(conv2d(x, conv_weights[0], stride=1, padding=0))
    x = max_pool2d(x, kernel_size=2, stride=2)
    x = dropout(x, p_conv1)
    
    x = rectify(conv2d(x, conv_weights[1], stride=1, padding=0))
    x = max_pool2d(x, kernel_size=2, stride=2)
    x = dropout(x, p_conv2)
    
    x = rectify(conv2d(x, conv_weights[2], stride=1, padding=0))
    x = max_pool2d(x, kernel_size=2, stride=2)
    x = dropout(x, p_conv3)
    
    x = x.reshape(x.shape[0], -1)
    x = rectify(x @ fc_weights[0])
    x = dropout(x, p_fc)
    logits = x @ fc_weights[1]
    return logits

In [ ]:
w_conv1 = init_weights((32, 1, 5, 5))
w_conv2 = init_weights((64, 32, 5, 5))
w_conv3 = init_weights((128, 64, 3, 3))

w_fc1 = init_weights((128, 625))
w_fc2 = init_weights((625, 10))

conv_params = [w_conv1, w_conv2, w_conv3]
fc_params = [w_fc1, w_fc2]
all_params = conv_params + fc_params

optimizer = RMSprop(params=all_params, lr=1e-3)
dropout_probs = [0.2, 0.5, 0.5, 0.5]

print("CNN weights initialized.")

In [ ]:
def evaluate_cnn_loss(dataloader):
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)  # shape already (batch, 1, 28, 28)
            y = y.to(device)
            # Disable dropout by passing p=0.0 for all
            logits = conv_model(x, conv_params, fc_params, [0.0, 0.0, 0.0, 0.0])
            loss = cross_entropy(logits, y, reduction="sum")
            total_loss += float(loss)
            total_samples += y.shape[0]
    return total_loss / total_samples

In [ ]:
train_loss_cnn = []
test_loss_cnn = []
eval_epochs_cnn = []

def evaluate_cnn_loss(dataloader):
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)
            logits = conv_model(x, conv_params, fc_params, [0.0, 0.0, 0.0, 0.0])
            loss = cross_entropy(logits, y, reduction="sum")
            total_loss += float(loss)
            total_samples += y.shape[0]
    return total_loss / total_samples

print("Training Task 4 (CNN)...")
for epoch in range(n_epochs + 1):
    for idx, batch in enumerate(train_dataloader):
        x, y = batch
        x = x.to(device)
        y = y.to(device)
        logits = conv_model(x, conv_params, fc_params, dropout_probs)
        optimizer.zero_grad()
        loss = cross_entropy(logits, y, reduction="mean")
        loss.backward()
        optimizer.step()
    
    if epoch % 10 == 0:
        train_loss_cnn.append(evaluate_cnn_loss(train_dataloader))
        test_loss_cnn.append(evaluate_cnn_loss(test_dataloader))
        eval_epochs_cnn.append(epoch)
        print(f"Epoch {epoch:3d}: Train Loss = {train_loss_cnn[-1]:.2e}, Test Loss = {test_loss_cnn[-1]:.2e}")

plt.figure(figsize=(8, 5))
plt.plot(eval_epochs_cnn, train_loss_cnn, 'o-', label="Train (CNN)", markersize=4)
plt.plot(eval_epochs_cnn, test_loss_cnn, 's-', label="Test (CNN)", markersize=4)
plt.title("Task 4: Convolutional Network - Train and Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

In [ ]:
test_iter = iter(test_dataloader)
sample_x, sample_y = next(test_iter)
sample_x = sample_x[0:1].to(device)

with torch.no_grad():
    conv1_out = rectify(conv2d(sample_x, w_conv1, stride=1, padding=0))

fig, axes = plt.subplots(2, 3, figsize=(12, 6))

for i in range(3):
    filter_img = w_conv1[i, 0].cpu().numpy()
    axes[0, i].imshow(filter_img, cmap='gray')
    axes[0, i].set_title(f"Filter {i+1}")
    axes[0, i].axis('off')
    
    feature_map = conv1_out[0, i].cpu().numpy()
    axes[1, i].imshow(feature_map, cmap='gray')
    axes[1, i].set_title(f"Feature map {i+1}")
    axes[1, i].axis('off')

plt.suptitle("First Convolutional Layer: Filters and Outputs")
plt.tight_layout()
plt.show()

In [ ]:
print("\n========== FINAL TEST LOSS COMPARISON ==========")
print(f"Task 1 (Plain ReLU)          : {test_loss_plain[-1]:.2e}")
print(f"Task 2 (Dropout)             : {test_loss_dropout[-1]:.2e}")
print(f"Task 3 (Parametric ReLU)     : {test_loss_prelu[-1]:.2e}")
print(f"Task 4 (Convolutional)       : {test_loss_cnn[-1]:.2e}")
print("=================================================")

plt.figure(figsize=(10, 6))
plt.plot(eval_epochs_plain, test_loss_plain, 'o-', label="Plain ReLU", markersize=4)
plt.plot(eval_epochs_dropout, test_loss_dropout, 's-', label="Dropout", markersize=4)
plt.plot(eval_epochs_prelu, test_loss_prelu, '^-', label="Parametric ReLU", markersize=4)
plt.plot(eval_epochs_cnn, test_loss_cnn, 'd-', label="CNN", markersize=4)
plt.title("Test Loss Comparison Across All Models")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()